In [25]:
#!/usr/bin/env python3
import torch
import numpy as np
import struct
import os

torch.set_default_dtype(torch.float64)
np.set_printoptions(threshold=np.inf)
np.set_printoptions(linewidth=np.inf)
model = torch.jit.load('./model.pt')

In [26]:
def format_array_content(content, max_len=1900):
    content = content.strip()
    if len(content) <= max_len:
        return content
    lines = []
    while len(content) > 0:
        if len(content) <= max_len:
            lines.append(content)
            break
        break_pos = content.rfind(',', 0, max_len)
        if break_pos == -1:
            break_pos = max_len
        lines.append(content[:break_pos+1] + "&")
        content = "   " + content[break_pos+1:].lstrip()
    return "\n".join(lines)

In [27]:
file = open("ml_classifier.f90", "w")
print("!> ML-Classifier\n!> Provides the architecture for the neural network and the weights/biases\n!> !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n!> !!!!!!!!!!!!! DO NOT MODIFY -- This module is automatically generated\n!> !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n!> Use generate_fortran_classifier.ipynb to generate this file for a given PyTorch model\n!> Class ids:\n!>    0: No classification (central cell is not mixed)\n!>    1: Well-resolved interface\n!>    2: Ligament\n!>    3: Droplet\n!>    4: Sheet/film\n!>    5: Ligament end\n!>    6: Sheet end\n",file=file)
print("module ml_classifier",file=file)
print("   use precision, only: WP\n   implicit none",file=file)
print("   integer :: idx", file=file)
print("   integer,  parameter ::   N = 5", file=file)
print("   integer,  parameter :: cid = N/2+1", file=file)
print("   real(WP), parameter :: epsilon_connect = 1.0e-12_WP", file=file)
print("   integer,  parameter :: n6(3, 6) = reshape([1,0,0,-1,0,0,0,1,0,0,-1,0,0,0,1,0,0,-1], [3, 6])", file=file)

param_info = []
count = 0
hidden = 0
sizes = []

for param in model.parameters():
    count = count + 1
    name = ""
    if count == 3:
        hidden = param.transpose(0,1).size()[0] #Assumes each hidden layer has same number of neurons
    
    if count%2 != 0:
        name = "lay" + str(int(count/2)+1) + "_weight"
        size1 = param.transpose(0,1).size()[0]
        size2 = param.transpose(0,1).size()[1]
        print(f"   real(WP), dimension({size1},{size2}), save :: {name}", file=file)
        param_info.append({'name': name, 'type': 'weight', 'data': param.detach().numpy()})
    else:
        name = "lay" + str(int((count-1)/2)+1) + "_bias"
        size1 = param.size()[0]
        sizes.append(size1)
        print(f"   real(WP), dimension({size1}), save :: {name}", file=file)
        param_info.append({'name': name, 'type': 'bias', 'data': param.detach().numpy()})

bias_size = 1000
for info in param_info:
    name = info['name']
    data = info['data']
    if info['type'] == 'weight':
        num_rows = data.shape[1]
        num_cols = data.shape[0]
        for j in range(num_cols):
            col_data = data[j, :]
            values_str = np.array2string(col_data, separator=', ')[1:-1].strip()
            formatted_values = format_array_content(values_str)
            var_list_str = f"({name}(idx, {j+1}), idx=1, {num_rows})"
            print(f"   DATA {var_list_str} /&\n   {formatted_values}/", file=file)
    elif info['type'] == 'bias':
        total_size = data.shape[0]
        for i in range(0, total_size, bias_size):
            start_index = i
            end_index = min(i + bias_size, total_size)
            chunk_data = data[start_index:end_index]
            values_str = np.array2string(chunk_data, separator=', ')[1:-1].strip()
            formatted_values = format_array_content(values_str)
            var_list_str = f"({name}(idx), idx={start_index+1}, {end_index})"
            print(f"   DATA {var_list_str} /&\n   {formatted_values}/", file=file)

print("\n   contains\n   integer function get_class(vfrac, liq_bary) result(predicted_class)\n      implicit none", file=file)
print("      real(WP), dimension(1:N,1:N,1:N), intent(inout) :: vfrac", file=file)
print("      real(WP), dimension(1:N,1:N,1:N,1:3), intent(inout) :: liq_bary", file=file)
print("      real(WP), dimension(N*N*N*4) :: flattened_state", file=file)
print("      real(WP), dimension(6) :: logits", file=file)

for i in range(len(sizes)-1): 
    print("      real(WP), dimension(" + str(sizes[i]) + ") :: tmparr" + str(i+1), file=file)
print("      predicted_class = 0", file=file)
print("      if (vfrac(cid, cid, cid) < epsilon_connect) return", file=file)
print("      call preprocess_and_flatten(vfrac, liq_bary, flattened_state)", file=file)
print("      tmparr"+str(1)+"=max(0.0_WP,matmul(flattened_state,lay1_weight)+lay1_bias)",file=file)
i=-1
for i in range(int((count-1)/2)-1):
    name1 = "lay" + str(i+2) + "_weight"
    name2 = "lay" + str(i+2) + "_bias"
    print("      tmparr"+str(i+2)+"=max(0.0_WP,matmul(tmparr"+str(i+1)+"," + name1 + ")+" + name2 + ")",file=file)
name1 = "lay" + str(i+3) + "_weight"
name2 = "lay" + str(i+3) + "_bias"
print("      logits=matmul(tmparr"+str(i+2)+"," + name1 + ")+" + name2,file=file)
print("      predicted_class = maxloc(logits, dim=1)",file=file)
print("   end function get_class", file=file)

reflect_subroutines = """
   subroutine reflect(vfrac, liq_bary, dir)
      real(WP), dimension(1:N,1:N,1:N), intent(inout) :: vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3), intent(inout) :: liq_bary
      integer, intent(in) :: dir
      real(WP), dimension(1:N,1:N,1:N) :: tmp_vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3) :: tmp_liq_bary
      integer  :: i, j, k, mirror_i, mirror_j, mirror_k
      ! Copy the original arrays to temporary arrays
      tmp_vfrac    = vfrac
      tmp_liq_bary = liq_bary
      do i=1,N; do j=1,N; do k=1,N
         mirror_i = merge(N+1-i,i,dir.eq.0)
         mirror_j = merge(N+1-j,j,dir.eq.1)
         mirror_k = merge(N+1-k,k,dir.eq.2)
         vfrac(i,j,k) = tmp_vfrac(mirror_i,mirror_j,mirror_k)
         if (dir.eq.0) then
            liq_bary(i,j,k,1) = -tmp_liq_bary(mirror_i,mirror_j,mirror_k,1)
         else
            liq_bary(i,j,k,1) =  tmp_liq_bary(mirror_i,mirror_j,mirror_k,1)
         end if
         if (dir.eq.1) then
            liq_bary(i,j,k,2) = -tmp_liq_bary(mirror_i,mirror_j,mirror_k,2)
         else
            liq_bary(i,j,k,2) =  tmp_liq_bary(mirror_i,mirror_j,mirror_k,2)
         end if
         if (dir.eq.2) then
            liq_bary(i,j,k,3) = -tmp_liq_bary(mirror_i,mirror_j,mirror_k,3)
         else
            liq_bary(i,j,k,3) =  tmp_liq_bary(mirror_i,mirror_j,mirror_k,3)
         end if
      end do; end do; end do
   end subroutine reflect
 
   subroutine permute(vfrac, liq_bary, dir)
      real(WP), dimension(1:N,1:N,1:N), intent(inout) :: vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3), intent(inout) :: liq_bary
      integer, intent(in) :: dir
      real(WP), dimension(1:N,1:N,1:N) :: tmp_vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3) :: tmp_liq_bary
      integer  :: i,j,k
      ! Copy the original arrays to temporary arrays
      tmp_vfrac    = vfrac
      tmp_liq_bary = liq_bary
      if (dir.eq.1) then
         do i=1,N; do j=1,N; do k=1,N
            vfrac(i,j,k)      = tmp_vfrac(k,i,j)
            liq_bary(i,j,k,1) = tmp_liq_bary(k,i,j,2)
            liq_bary(i,j,k,2) = tmp_liq_bary(k,i,j,3)
            liq_bary(i,j,k,3) = tmp_liq_bary(k,i,j,1)
         end do; end do; end do
      else if (dir.eq.2) then
         do i=1,N; do j=1,N; do k=1,N
            vfrac(i,j,k)      = tmp_vfrac(j,k,i)
            liq_bary(i,j,k,1) = tmp_liq_bary(j,k,i,3)
            liq_bary(i,j,k,2) = tmp_liq_bary(j,k,i,1)
            liq_bary(i,j,k,3) = tmp_liq_bary(j,k,i,2)
         end do; end do; end do
      end if
   end subroutine permute
 
   subroutine swap_xy(vfrac, liq_bary)
      real(WP), dimension(1:N,1:N,1:N), intent(inout) :: vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3), intent(inout) :: liq_bary
      real(WP), dimension(1:N,1:N,1:N) :: tmp_vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3) :: tmp_liq_bary
      integer  :: i,j,k
      ! Copy the original arrays to temporary arrays
      tmp_vfrac    = vfrac
      tmp_liq_bary = liq_bary
      do i=1,N; do j=1,N; do k=1,N
         vfrac(i,j,k)      = tmp_vfrac(j,i,k)
         liq_bary(i,j,k,1) = tmp_liq_bary(j,i,k,2)
         liq_bary(i,j,k,2) = tmp_liq_bary(j,i,k,1)
         liq_bary(i,j,k,3) = tmp_liq_bary(j,i,k,3)
      end do; end do; end do
   end subroutine swap_xy
 
   pure integer function largest_value_index(v0,v1,v2) result(largest_idx)
      real(WP), intent(in) :: v0,v1,v2
      real(WP) :: largest
      largest = v0
      largest_idx = 0
      if (v1 > largest) then
         largest = v1
         largest_idx = 1
      end if
      if (v2 > largest) then
         largest = v2
         largest_idx = 2
      end if
   end function largest_value_index
 
   subroutine preprocess_and_flatten(vfrac, liq_bary, flattened_state)
      real(WP), dimension(1:N,1:N,1:N), intent(inout) :: vfrac
      real(WP), dimension(1:N,1:N,1:N,1:3), intent(inout) :: liq_bary
      real(WP), dimension(N*N*N*4), intent(inout) :: flattened_state
      logical,  dimension(1:N,1:N,1:N) :: visited
      integer,  dimension(3, N*N*N) :: q
      integer  :: qsize, qi
      integer  :: i, j, k, m, ni, nj, nk
      integer  :: largest_idx
      real(WP), dimension(1:3) :: global_liq_bary
      real(WP) :: total_volume
      real(WP) :: tmp
      integer  :: n_inputs, pos
 
      ! ---- initialize visited & seed with central cell -----------------
      ! NOTE: `cid` must be redefined elsewhere as cid = N/2 + 1
      visited = .false.
      qsize = 0
      visited(cid, cid, cid) = .true.
      qsize = qsize + 1
      q(:, qsize) = [cid, cid, cid]
      flattened_state = 0.0_WP

      ! ---- flood fill (BFS) --------------------------------------------
      qi = 1
      do while (qi <= qsize)
         i = q(1, qi); j = q(2, qi); k = q(3, qi)
         do m = 1, 6
            ni = i + n6(1, m)
            nj = j + n6(2, m)
            nk = k + n6(3, m)
            if (ni < 1 .or. ni > N .or. &
               nj < 1 .or. nj > N .or. &
               nk < 1 .or. nk > N) cycle
            if (visited(ni, nj, nk)) cycle
            if (vfrac(ni, nj, nk) > epsilon_connect) then
               visited(ni, nj, nk) = .true.
               qsize = qsize + 1
               q(:, qsize) = [ni, nj, nk]
            end if
         end do
         qi = qi + 1
      end do
 
      ! ---- zero everything not connected --------------------------------
      do i=1,N; do j=1,N; do k=1,N
         if (visited(i, j, k)) cycle
         if (vfrac(i, j, k) > 0.0_WP) then
            vfrac(i, j, k)       = 0.0_WP
            liq_bary(i, j, k, 1) = 0.0_WP
            liq_bary(i, j, k, 2) = 0.0_WP
            liq_bary(i, j, k, 3) = 0.0_WP
         end if
      end do; end do; end do
 
      ! ---- compute global liquid centroid in stencil --------------------
      global_liq_bary = 0.0_WP
      total_volume    = 0.0_WP
      do i=1,N; do j=1,N; do k=1,N
         total_volume = total_volume + vfrac(i, j, k)
         global_liq_bary(1) = global_liq_bary(1) + liq_bary(i, j, k, 1)
         global_liq_bary(2) = global_liq_bary(2) + liq_bary(i, j, k, 2)
         global_liq_bary(3) = global_liq_bary(3) + liq_bary(i, j, k, 3)
      end do; end do; end do
      if (total_volume.ne.0.0_WP) global_liq_bary = global_liq_bary / total_volume
 
      ! ---- reflect into first octant: ensure cx,cy,cz >= 0 --------------
      if (global_liq_bary(1) < 0.0_WP) then
         call reflect(vfrac, liq_bary, 0)
         global_liq_bary(1) = -global_liq_bary(1)
      end if
      if (global_liq_bary(2) < 0.0_WP) then
         call reflect(vfrac, liq_bary, 1)
         global_liq_bary(2) = -global_liq_bary(2)
      end if
      if (global_liq_bary(3) < 0.0_WP) then
         call reflect(vfrac, liq_bary, 2)
         global_liq_bary(3) = -global_liq_bary(3)
      end if
 
      largest_idx = largest_value_index(global_liq_bary(3), global_liq_bary(1), &
                                 global_liq_bary(2))
      if (largest_idx == 1) then
         call permute(vfrac, liq_bary, 1)
         ! (cx,cy,cz) -> (cy,cz,cx)
         tmp = global_liq_bary(1)
         global_liq_bary(1) = global_liq_bary(2)
         global_liq_bary(2) = global_liq_bary(3)
         global_liq_bary(3) = tmp
      else if (largest_idx == 2) then
         call permute(vfrac, liq_bary, 2)
         ! (cx,cy,cz) -> (cz,cx,cy)
         tmp = global_liq_bary(1)
         global_liq_bary(1) = global_liq_bary(3)
         global_liq_bary(3) = global_liq_bary(2)
         global_liq_bary(2) = tmp
      end if
 
      if (global_liq_bary(2) > global_liq_bary(1)) then
         ! count small differences (commented +1e-8 tolerance in original)
         call swap_xy(vfrac, liq_bary)
         tmp = global_liq_bary(1)
         global_liq_bary(1) = global_liq_bary(2)
         global_liq_bary(2) = tmp
      end if
 
      ! ---- flatten stencil into 1D vector --------------------------------
      ! size = vfrac + (mx, my, mz) for each cell
      pos = 0
      do i=1,N; do j=1,N; do k=1,N
         pos = pos + 1
         flattened_state(pos) = vfrac(i, j, k)
         pos = pos + 1
         flattened_state(pos) = liq_bary(i, j, k, 1)
         pos = pos + 1
         flattened_state(pos) = liq_bary(i, j, k, 2)
         pos = pos + 1
         flattened_state(pos) = liq_bary(i, j, k, 3)
      end do; end do; end do
   end subroutine preprocess_and_flatten
   """

print(reflect_subroutines, file=file)
print("end module ml_classifier", file=file)

file.close()
